In [8]:
import cv2
import base64
import requests
import uiautomator2 as u2
DEVICE_SERIAL = "emulator-5556"
OCR_URL = "http://127.0.0.1:5001/ocr"   # 你的 OCR Server /ocr
TIMEOUT = 25
# ========= OCR 呼叫 / 影像處理 =========
def preprocess_for_ocr(roi_bgr):
    # 放大 + 輕微去噪（你現在效果已經不錯，這段是加穩定）
    h, w = roi_bgr.shape[:2]
    roi = cv2.resize(roi_bgr, (w * 2, h * 2), interpolation=cv2.INTER_CUBIC)
    roi = cv2.GaussianBlur(roi, (3, 3), 0)
    return roi

def to_b64_jpg(img_bgr, quality=92) -> str:
    ok, buf = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not ok:
        raise RuntimeError("cv2.imencode failed")
    return base64.b64encode(buf.tobytes()).decode("ascii")

def call_ocr(img_bgr):
    b64 = to_b64_jpg(img_bgr)
    resp = requests.post(OCR_URL, json={"image": b64}, timeout=TIMEOUT)
    print(resp.json())
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success", False):
        raise RuntimeError(f"OCR server error: {data}")
    return data.get("ocr_results", []) or []
d = u2.connect(DEVICE_SERIAL)
screenshot = d.screenshot(format="opencv")
#當前的 
# 153 385 470 525
# roi = screenshot[385:525, 153:470]
# roi_preprocessed = preprocess_for_ocr(roi)
# ocr_results = call_ocr(roi_preprocessed)
# print("OCR 結果:", ocr_results)
#新開的
# 155 645 458 755
roi = screenshot[645:755, 155:458]
roi_preprocessed = preprocess_for_ocr(roi)
ocr_results = call_ocr(roi_preprocessed)
print("OCR 結果:", ocr_results)
import re
import pandas as pd
from typing import Optional, List, Dict, Tuple

# =========================
# 1) 可維護的「資料表」
# =========================

# A. OCR 常見誤判修正（只做「片語」替換，避免誤傷）
REPLACEMENTS: List[Tuple[str, str]] = [
    (" ", ""),
    ("\n", ""),
    ("\t", ""),

    ("攻擎", "攻擊"),
    ("擎量", "擊暈"),
    ("眩量", "暈眩"),     # 你遊戲若用「眩暈」就改這行
    ("擊量", "擊暈"),

    ("普攻使方", "普攻使敵方"),
    ("额外", "額外"),
    ("额", "額"),
    ("永恒", "永恆"),

    ("暴馨", "暴擊"),
    ("暴撃", "暴擊"),
    ("技能爆擊", "技能暴擊"),
    ("爆擊", "暴擊"),

    ("连撃", "連擊"),
    ("連撃", "連擊"),
    ("回复", "回復"),
]

# B. 詞條字典（canonical -> code + aliases）
# 你想支援更多詞條/同義字，只要加 aliases
AFFIX_DICT: Dict[str, Dict[str, List[str] or str]] = {
    "技能暴擊": {"code": "技", "aliases": ["技能暴擊", "技能爆擊", "技暴", "技能暴"]},
    "反擊":     {"code": "反", "aliases": ["反擊", "反"]},
    "暴擊":     {"code": "爆", "aliases": ["暴擊", "爆擊", "暴", "爆"]},
    "連擊":     {"code": "連", "aliases": ["連擊", "連"]},
    "擊暈":     {"code": "暈", "aliases": ["擊暈", "暈眩", "眩暈", "暈", "眩"]},
    "閃避":     {"code": "閃", "aliases": ["閃避", "閃"]},
    "回復":     {"code": "回", "aliases": ["回復", "回血", "回"]},
}

# C. 不允許的 combo（任何順序都算）
UNWANTED_COMBOS = {
    ('技', '反'), ('技', '連'), ('技', '爆'), ('技', '閃'),
    ('連', '暈'), ('連', '反'), ('連', '回'),
    ('暈', '閃'), ('暈', '爆'), ('暈', '反'),
    ('反', '閃'), ('爆', '回'), ('爆', '暈')
}

# D. combo 正規化（用 set 當 key）
CANONICAL_PAIR = {
    frozenset({"閃", "爆"}): "連閃",   # 你原本把閃爆歸連閃
    frozenset({"連", "暈"}): "連暈",
    frozenset({"技", "回"}): "技回",
    frozenset({"反", "爆"}): "反爆",
}

# E. 額外字串替換（保留你原本那套）
PAIR_REWRITE = {
    '爆閃': '連閃', '閃閃': '連閃', '閃爆': '連閃', '閃連': '連閃',
    '爆連': '連爆', '回技': '技回', '回閃': '閃回', '爆反': '反爆',
    '回暈': '暈回', '回反': '反回', '暈技': '技暈'
}

# 如果你還是想保留「禁止組合」的快速查表（可選）
AFFIX_KEYS = list(AFFIX_DICT.keys())


# =========================
# 2) 固定邏輯（通常不需要改）
# =========================

def normalize_text(text: str) -> str:
    if not text:
        return ""
    for a, b in REPLACEMENTS:
        text = text.replace(a, b)
    return text

# alias -> code（長的先匹配）
ALIAS_TO_CODE: List[Tuple[str, str]] = []
for canon, info in AFFIX_DICT.items():
    for alias in info["aliases"]:
        ALIAS_TO_CODE.append((alias, info["code"]))
ALIAS_TO_CODE.sort(key=lambda x: len(x[0]), reverse=True)

def text_to_skill_code(text: str) -> Optional[str]:
    t = normalize_text(text)
    for alias, code in ALIAS_TO_CODE:
        if alias and alias in t:
            return code
    return None

def normalize_combo(combo: str) -> str:
    if len(combo) != 2:
        return combo
    # set-based canonical
    key = frozenset(combo)
    if key in CANONICAL_PAIR:
        combo = CANONICAL_PAIR[key]
    # string rewrite fallback
    return PAIR_REWRITE.get(combo, combo)

def is_unwanted_combo(combo: str) -> bool:
    if len(combo) != 2:
        return False
    a, b = combo[0], combo[1]
    return (a, b) in UNWANTED_COMBOS or (b, a) in UNWANTED_COMBOS


# =========================
# 3) 面板/詞條抽取 + combo 整合輸出
# =========================

def extract_panel_and_entries(ocr_list):
    """
    回傳:
      panel: {'生': int, '攻擊': int, '防禦': int} (有抓到才會出現 key)
      entries: [{'詞條': str|None, '%': float}, ...]  (同框/分框都會抓)
      items: normalize 後的 OCR items（方便 debug）
    """
    items = [
        {"text": normalize_text(x.get("text", "")),
         "bbox": x.get("bbox"),
         "score": x.get("score", 1.0)}
        for x in ocr_list
        if x.get("text") is not None and x.get("bbox") is not None
    ]

    def center_y(b):
        return (b[1] + b[3]) / 2

    # ---------- A) 面板：生命/攻擊/防禦 ----------
    panel: Dict[str, int] = {}

    # (1) 同框：生命056780
    for it in items:
        m = re.search(r"(生|攻擊|防禦)(\d+)", it["text"])
        if m:
            panel[m.group(1)] = int(m.group(2))

    # (2) 分框：label + 數字（同列右側最近）
    num_items = [it for it in items if re.fullmatch(r"\d+", it["text"])]
    label_items = [it for it in items if it["text"] in ["生", "攻擊", "防禦"]]

    for lab in label_items:
        if lab["text"] in panel:
            continue
        candidates = []
        for num in num_items:
            if abs(center_y(num["bbox"]) - center_y(lab["bbox"])) <= 12 and num["bbox"][0] >= lab["bbox"][2] - 5:
                dx = num["bbox"][0] - lab["bbox"][2]
                candidates.append((dx, num))
        if candidates:
            candidates.sort(key=lambda t: t[0])
            panel[lab["text"]] = int(candidates[0][1]["text"])

    # ---------- B) 詞條 + % ----------
    entries = []
    used_idx = set()

    # (1) 同框：技能暴擊3.32%
    for i, it in enumerate(items):
        m = re.search(r"(.+?)(\d+(?:\.\d+)?)%", it["text"])
        if m and m.group(1) and not re.fullmatch(r"[\d\.]+", m.group(1)):
            entries.append({"詞條": m.group(1), "%": float(m.group(2))})
            used_idx.add(i)

    # (2) 分框：詞條一格 + % 一格（同列左側最近）
    def is_term_text(t: str) -> bool:
        if t in ["生命", "攻擊", "防禦"]:
            return False
        if re.fullmatch(r"\d+", t):
            return False
        if re.fullmatch(r"\d+(?:\.\d+)?%", t):
            return False
        if re.search(r"\d", t) and "%" in t:
            return False
        return True

    percent_items = [(i, it) for i, it in enumerate(items)
                     if i not in used_idx and re.fullmatch(r"\d+(?:\.\d+)?%", it["text"])]

    term_items = [(i, it) for i, it in enumerate(items) if is_term_text(it["text"])]

    for pi, p in percent_items:
        candidates = []
        for ti, t in term_items:
            if abs(center_y(t["bbox"]) - center_y(p["bbox"])) <= 12 and t["bbox"][2] <= p["bbox"][0] + 5:
                dx = p["bbox"][0] - t["bbox"][2]
                candidates.append((dx, ti, t))
        if candidates:
            candidates.sort(key=lambda x: x[0])
            _, ti, t = candidates[0]
            entries.append({"詞條": t["text"], "%": float(p["text"].rstrip("%"))})
            used_idx.add(pi)
            used_idx.add(ti)
        else:
            entries.append({"詞條": None, "%": float(p["text"].rstrip("%"))})
            used_idx.add(pi)

    # 去重
    entries = pd.DataFrame(entries).drop_duplicates().to_dict("records")
    return panel, entries, items


def build_combo_from_entries(entries) -> str:
    """
    以 entries 的詞條優先產 combo（比掃所有 OCR 更準）
    """
    skills = []
    for e in entries:
        t = e.get("詞條") or ""
        code = text_to_skill_code(t)
        if code and code not in skills:
            skills.append(code)
        if len(skills) == 2:
            break
    return "".join(skills)


def parse_ocr(ocr_results):
    """
    一次輸出你最常用的所有結果：
      - panel
      - entries（詞條+%）
      - combo_raw / combo_norm / unwanted
      - debug_items（normalize 後）
    """
    panel, entries, items = extract_panel_and_entries(ocr_results)

    combo_raw = build_combo_from_entries(entries)
    # 若 entries 太少（例如 % 沒抓到），退回掃全 OCR
    if len(combo_raw) < 2:
        skills = []
        for it in items:
            code = text_to_skill_code(it["text"])
            if code and code not in skills:
                skills.append(code)
            if len(skills) == 2:
                break
        combo_raw = "".join(skills)

    combo_norm = normalize_combo(combo_raw)
    unwanted = is_unwanted_combo(combo_norm if len(combo_norm) == 2 else combo_raw)

    return {
        "panel": panel,
        "entries": entries,
        "combo_raw": combo_raw,
        "combo_norm": combo_norm,
        "unwanted": unwanted,
        "debug_items": items,  # 需要時拿來看 normalize 後的 OCR 字串
    }


# =========================
# 4) 用法
# =========================
result = parse_ocr(ocr_results)
print(result["panel"])
print(result["entries"])
print(result["combo_raw"], result["combo_norm"], result["unwanted"])


{'meta': {'h': 220, 'w': 606}, 'ocr_results': [{'bbox': [275, 7, 349, 49], 'bbox_rel': [0.4537953795379538, 0.031818181818181815, 0.5759075907590759, 0.22272727272727272], 'score': 0.6222478747367859, 'text': '反撃'}, {'bbox': [29, 10, 289, 47], 'bbox_rel': [0.04785478547854786, 0.045454545454545456, 0.4768976897689769, 0.21363636363636362], 'score': 0.9178669452667236, 'text': '生308006分'}, {'bbox': [484, 12, 549, 51], 'bbox_rel': [0.7986798679867987, 0.05454545454545454, 0.905940594059406, 0.2318181818181818], 'score': 0.9210702180862427, 'text': '7%'}, {'bbox': [26, 57, 100, 98], 'bbox_rel': [0.0429042904290429, 0.2590909090909091, 0.16501650165016502, 0.44545454545454544], 'score': 0.7335144877433777, 'text': '攻擊'}, {'bbox': [145, 56, 243, 98], 'bbox_rel': [0.23927392739273928, 0.2545454545454545, 0.400990099009901, 0.44545454545454544], 'score': 0.9993363618850708, 'text': '7214'}, {'bbox': [239, 53, 575, 93], 'bbox_rel': [0.3943894389438944, 0.2409090909090909, 0.9488448844884488, 0

In [ ]:
d=u2.connect("emulator-5554")
img = d.screenshot(format="opencv")
print(img[585,48])  # [ True  True  True]
if not img[720,30].tolist() == [80, 95, 111] and not img[477,148].tolist() == [58, 73, 129]:
    print("Not matched")   
#在上面繪製一個點 at 83,580
cv2.circle(img, (83,580), 5, (0,0,255), -1)
cv2.imshow("screenshot", img)   
cv2.waitKey(0)
if [115, 109, 114]== img[580,83].tolist():
    print("Matched")

    

[113 110 112]


In [85]:
#點擊cv2的圖片 顯示該圖片點擊位置的顏色和座標
def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        color = img[y, x].tolist()
        print(f"Clicked at ({x}, {y}), Color: {color}")
img = d.screenshot(format="opencv")
cv2.imshow("screenshot", img)
cv2.setMouseCallback("screenshot", on_mouse)
cv2.waitKey(0)

Clicked at (332, 564), Color: [58, 65, 198]
Clicked at (212, 564), Color: [58, 65, 198]
Clicked at (246, 2), Color: [142, 146, 140]
Clicked at (288, 474), Color: [198, 224, 230]


32

In [ ]:
# Clicked at (127, 529), Color: [178, 209, 218]
# Clicked at (264, 585), Color: [178, 209, 218]
# Clicked at (119, 707), Color: [178, 209, 218]
# Clicked at (218, 793), Color: [58, 65, 198]
# Clicked at (417, 794), Color: [42, 155, 111]
# Clicked at (34, 726), Color: [76, 91, 107]
# Clicked at (523, 537), Color: [109, 122, 130]
#根據座標點 判斷當前狀態 
img = d.screenshot(format="opencv")
if img[529,127].tolist() == [178, 209, 218] and img[585,264].tolist() == [178, 209, 218] and img[707,119].tolist() == [178, 209, 218] and img[793,218].tolist() == [58, 65, 198] and img[794,417].tolist() == [42, 155, 111] and img[726,34].tolist() == [76, 91, 107] and img[537,523].tolist() == [109, 122, 130]:
    print("當前在開到裝備")
elif img[564,332].tolist() == [58, 65, 198] and img[564,212].tolist() == [58, 65, 198]:
    print("當前在全部出售頁面")

當前在全部出售頁面


In [86]:
if img[564,332].tolist() == [58, 65, 198] and img[564,212].tolist() == [58, 65, 198]:
    print("當前在切磋頁面")

當前在切磋頁面
